# TUGAS MANDIRI DATA WRANGLING

**Nama:** Marcella Grace Sensiana Dhey  
**NIM:** 25200001  
**Kelas:** 3A  
**Mata Kuliah:** Data Wrangling

In [54]:
## BAGIAN A
### Analisis Konseptual & Tata Kelola Data

#### 1. Pemetaan Kebutuhan Bisnis menjadi Kebutuhan Teknis Data

#### 2. Matriks Peran RACI

### 1. Pemetaan Kebutuhan Bisnis menjadi Kebutuhan Teknis Data

Kebutuhan bisnis dari Head of Business Strategy adalah menganalisis performa
kampanye kupon diskon akhir bulan. Informasi yang dibutuhkan meliputi jumlah
transaksi yang menggunakan kupon, profil loyalitas pelanggan yang menggunakan
kupon, serta status pengiriman pesanan oleh mitra logistik.

Untuk memenuhi kebutuhan tersebut, diperlukan integrasi data dari tiga sumber,
yaitu data transaksi dari Divisi Kasir/POS, data pelanggan dari Divisi CRM /
Membership, dan data pengiriman dari mitra ekspedisi logistik.

| Kebutuhan Informasi Bisnis | Atribut Data | Tipe Data | Sumber Data |
|---|---|---|---|
| Mengidentifikasi transaksi | `order_id` | string | CSV POS |
| Mengidentifikasi pelanggan | `customer_id` | string | CSV POS / Database CRM |
| Mengetahui tanggal transaksi | `order_date` | datetime | CSV POS |
| Mengetahui nilai transaksi | `total_amount` | float | CSV POS |
| Mengetahui penggunaan kupon | `coupon_code` | string | CSV POS |
| Mengetahui nama pelanggan | `customer_name` | string | Database CRM |
| Mengetahui status aktif pelanggan | `is_active` | integer/boolean | Database CRM |
| Mengetahui level loyalitas pelanggan | `membership_level` | string | Database CRM |
| Mengetahui jasa ekspedisi | `courier` | string | REST API Logistik |
| Mengetahui nomor resi | `tracking_number` | string | REST API Logistik |
| Mengetahui status pengiriman | `shipping_status` | string | REST API Logistik |

### 2. Matriks Peran RACI

RACI digunakan untuk menjelaskan pembagian tanggung jawab setiap pihak dalam
proses pengelolaan dan pengolahan data.

| Aktivitas | Head of Business Strategy | Data Management Office / Data Steward | Data Wrangling / Data Engineer | Business Intelligence Analyst |
|---|---|---|---|---|
| Menentukan kebutuhan bisnis | A | C | C | C |
| Menentukan definisi dan aturan data | C | A | R | C |
| Ekstraksi data CSV | I | C | R | I |
| Ekstraksi data database | I | C | R | I |
| Konsumsi REST API | I | C | R | I |
| Integrasi data multi-sumber | I | C | R | C |
| Validasi kualitas dan integritas data | C | A | R | C |
| Melakukan analisis data | I | C | C | R |
| Menyajikan hasil analisis | A | I | C | R |

**Keterangan:**

- **R (Responsible)**: pihak yang bertanggung jawab langsung dalam mengerjakan aktivitas.
- **A (Accountable)**: pihak yang memiliki tanggung jawab akhir terhadap hasil aktivitas.
- **C (Consulted)**: pihak yang memberikan masukan atau konsultasi.
- **I (Informed)**: pihak yang perlu mendapatkan informasi mengenai aktivitas atau hasilnya.

In [55]:
import pandas as pd
import sqlite3
import requests

In [56]:
data_transaksi = {
    "order_id": [
        "ORD001", "ORD002", "ORD003", "ORD004", "ORD005",
        "ORD006", "ORD007", "ORD008", "ORD009", "ORD010"
    ],
    "customer_id": [
        "C001", "C002", "C003", "C004", "C005",
        "C006", "C007", "C008", "C009", "C010"
    ],
    "order_date": [
        "2026-09-25", "2026-09-26", "2026-09-26", "2026-09-27",
        "2026-09-28", "2026-09-28", "2026-09-29", "2026-09-29",
        "2026-09-30", "2026-09-30"
    ],
    "total_amount": [
        150000, 225000, 175000, 320000, 125000,
        275000, 180000, 95000, 450000, 210000
    ],
    "coupon_code": [
        "AKHIRBULAN", None, "AKHIRBULAN", "AKHIRBULAN", None,
        "AKHIRBULAN", None, "AKHIRBULAN", "AKHIRBULAN", None
    ]
}

df_transaksi_awal = pd.DataFrame(data_transaksi)

df_transaksi_awal

,order_id,customer_id,order_date,total_amount,coupon_code
0,ORD001,C001,2026-09-25,150000,AKHIRBULAN
1,ORD002,C002,2026-09-26,225000,None
2,ORD003,C003,2026-09-26,175000,AKHIRBULAN
3,ORD004,C004,2026-09-27,320000,AKHIRBULAN
4,ORD005,C005,2026-09-28,125000,None
5,ORD006,C006,2026-09-28,275000,AKHIRBULAN
6,ORD007,C007,2026-09-29,180000,None
7,ORD008,C008,2026-09-29,95000,AKHIRBULAN
8,ORD009,C009,2026-09-30,450000,AKHIRBULAN
9,ORD010,C010,2026-09-30,210000,None


In [57]:
df_transaksi_awal.to_csv(
    "transaksi_penjualan.csv",
    index=False
)

print("File transaksi_penjualan.csv berhasil dibuat!")

File transaksi_penjualan.csv berhasil dibuat!


In [58]:
df_transaksi = pd.read_csv(
    "transaksi_penjualan.csv",
    dtype={"order_id": str},
    parse_dates=["order_date"]
)

df_transaksi.head()

,order_id,customer_id,order_date,total_amount,coupon_code
0,ORD001,C001,2026-09-25,150000,AKHIRBULAN
1,ORD002,C002,2026-09-26,225000,NaN
2,ORD003,C003,2026-09-26,175000,AKHIRBULAN
3,ORD004,C004,2026-09-27,320000,AKHIRBULAN
4,ORD005,C005,2026-09-28,125000,NaN


In [59]:
df_transaksi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      10 non-null     object        
 1   customer_id   10 non-null     object        
 2   order_date    10 non-null     datetime64[ns]
 3   total_amount  10 non-null     int64         
 4   coupon_code   6 non-null      object        
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 532.0+ bytes


In [60]:
# Membuat koneksi ke database SQLite
conn = sqlite3.connect("crm_membership.db")

print("Koneksi database berhasil!")

Koneksi database berhasil!


In [61]:
data_customer = {
    "customer_id": [
        "C001", "C002", "C003", "C004", "C005",
        "C006", "C007", "C008", "C009", "C010"
    ],
    "customer_name": [
        "Andi", "Budi", "Citra", "Dina", "Eka",
        "Fajar", "Gina", "Hadi", "Intan", "Joko"
    ],
    "membership_level": [
        "Gold", "Silver", "Platinum", "Gold", "Bronze",
        "Silver", "Gold", "Platinum", "Silver", "Gold"
    ],
    "is_active": [
        1, 1, 1, 1, 0,
        1, 1, 1, 1, 1
    ]
}

df_customer = pd.DataFrame(data_customer)

df_customer

,customer_id,customer_name,membership_level,is_active
0,C001,Andi,Gold,1
1,C002,Budi,Silver,1
2,C003,Citra,Platinum,1
3,C004,Dina,Gold,1
4,C005,Eka,Bronze,0
5,C006,Fajar,Silver,1
6,C007,Gina,Gold,1
7,C008,Hadi,Platinum,1
8,C009,Intan,Silver,1
9,C010,Joko,Gold,1


In [62]:
df_customer.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

print("Tabel customers berhasil dibuat!")

Tabel customers berhasil dibuat!


In [63]:
query = """
SELECT
    customer_id,
    customer_name,
    membership_level,
    is_active
FROM customers
WHERE is_active = 1
"""

df_customer_aktif = pd.read_sql_query(
    query,
    conn
)

df_customer_aktif

,customer_id,customer_name,membership_level,is_active
0,C001,Andi,Gold,1
1,C002,Budi,Silver,1
2,C003,Citra,Platinum,1
3,C004,Dina,Gold,1
4,C006,Fajar,Silver,1
5,C007,Gina,Gold,1
6,C008,Hadi,Platinum,1
7,C009,Intan,Silver,1
8,C010,Joko,Gold,1


In [64]:
url = "https://jsonplaceholder.typicode.com/todos"

response = requests.get(url)

print("Status code:", response.status_code)

Status code: 200


In [65]:
data_api = response.json()

print(type(data_api))
print(data_api[:3])

<class 'list'>
[{'userId': 1, 'id': 1, 'title': 'delectus aut autem', 'completed': False}, {'userId': 1, 'id': 2, 'title': 'quis ut nam facilis et officia qui', 'completed': False}, {'userId': 1, 'id': 3, 'title': 'fugiat veniam minus', 'completed': False}]


In [66]:
data_logistik = {
    "orders": [
        {
            "order_id": "ORD001",
            "shipping": {
                "courier": "JNE",
                "tracking_number": "JNE001",
                "status": "Delivered"
            }
        },
        {
            "order_id": "ORD002",
            "shipping": {
                "courier": "SiCepat",
                "tracking_number": "SCP002",
                "status": "In Transit"
            }
        },
        {
            "order_id": "ORD003",
            "shipping": {
                "courier": "J&T",
                "tracking_number": "JNT003",
                "status": "Delivered"
            }
        },
        {
            "order_id": "ORD004",
            "shipping": {
                "courier": "AnterAja",
                "tracking_number": "ANT004",
                "status": "Delivered"
            }
        },
        {
            "order_id": "ORD006",
            "shipping": {
                "courier": "JNE",
                "tracking_number": "JNE006",
                "status": "In Transit"
            }
        },
        {
            "order_id": "ORD008",
            "shipping": {
                "courier": "SiCepat",
                "tracking_number": "SCP008",
                "status": "Delivered"
            }
        },
        {
            "order_id": "ORD009",
            "shipping": {
                "courier": "J&T",
                "tracking_number": "JNT009",
                "status": "In Transit"
            }
        }
    ]
}

print("Data logistik berhasil dibuat.")

Data logistik berhasil dibuat.


In [67]:
df_logistik = pd.json_normalize(
    data_logistik["orders"]
)

df_logistik

,order_id,shipping.courier,shipping.tracking_number,shipping.status
0,ORD001,JNE,JNE001,Delivered
1,ORD002,SiCepat,SCP002,In Transit
2,ORD003,J&T,JNT003,Delivered
3,ORD004,AnterAja,ANT004,Delivered
4,ORD006,JNE,JNE006,In Transit
5,ORD008,SiCepat,SCP008,Delivered
6,ORD009,J&T,JNT009,In Transit


In [68]:
df_logistik = df_logistik.rename(columns={
    "shipping.courier": "courier",
    "shipping.tracking_number": "tracking_number",
    "shipping.status": "shipping_status"
})

df_logistik

,order_id,courier,tracking_number,shipping_status
0,ORD001,JNE,JNE001,Delivered
1,ORD002,SiCepat,SCP002,In Transit
2,ORD003,J&T,JNT003,Delivered
3,ORD004,AnterAja,ANT004,Delivered
4,ORD006,JNE,JNE006,In Transit
5,ORD008,SiCepat,SCP008,Delivered
6,ORD009,J&T,JNT009,In Transit


In [69]:
df_master_analisis = df_transaksi.merge(
    df_customer_aktif,
    on="customer_id",
    how="left"
)

df_master_analisis

,order_id,customer_id,order_date,total_amount,coupon_code,customer_name,membership_level,is_active
0,ORD001,C001,2026-09-25,150000,AKHIRBULAN,Andi,Gold,1.0
1,ORD002,C002,2026-09-26,225000,NaN,Budi,Silver,1.0
2,ORD003,C003,2026-09-26,175000,AKHIRBULAN,Citra,Platinum,1.0
3,ORD004,C004,2026-09-27,320000,AKHIRBULAN,Dina,Gold,1.0
4,ORD005,C005,2026-09-28,125000,NaN,NaN,NaN,NaN
5,ORD006,C006,2026-09-28,275000,AKHIRBULAN,Fajar,Silver,1.0
6,ORD007,C007,2026-09-29,180000,NaN,Gina,Gold,1.0
7,ORD008,C008,2026-09-29,95000,AKHIRBULAN,Hadi,Platinum,1.0
8,ORD009,C009,2026-09-30,450000,AKHIRBULAN,Intan,Silver,1.0
9,ORD010,C010,2026-09-30,210000,NaN,Joko,Gold,1.0


In [70]:
df_master_analisis = df_master_analisis.merge(
    df_logistik,
    on="order_id",
    how="left"
)

df_master_analisis

,order_id,customer_id,order_date,total_amount,coupon_code,customer_name,membership_level,is_active,courier,tracking_number,shipping_status
0,ORD001,C001,2026-09-25,150000,AKHIRBULAN,Andi,Gold,1.0,JNE,JNE001,Delivered
1,ORD002,C002,2026-09-26,225000,NaN,Budi,Silver,1.0,SiCepat,SCP002,In Transit
2,ORD003,C003,2026-09-26,175000,AKHIRBULAN,Citra,Platinum,1.0,J&T,JNT003,Delivered
3,ORD004,C004,2026-09-27,320000,AKHIRBULAN,Dina,Gold,1.0,AnterAja,ANT004,Delivered
4,ORD005,C005,2026-09-28,125000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ORD006,C006,2026-09-28,275000,AKHIRBULAN,Fajar,Silver,1.0,JNE,JNE006,In Transit
6,ORD007,C007,2026-09-29,180000,NaN,Gina,Gold,1.0,NaN,NaN,NaN
7,ORD008,C008,2026-09-29,95000,AKHIRBULAN,Hadi,Platinum,1.0,SiCepat,SCP008,Delivered
8,ORD009,C009,2026-09-30,450000,AKHIRBULAN,Intan,Silver,1.0,J&T,JNT009,In Transit
9,ORD010,C010,2026-09-30,210000,NaN,Joko,Gold,1.0,NaN,NaN,NaN


In [71]:
def buat_kamus_data(df):
    hasil = []

    for kolom in df.columns:
        hasil.append({
            "Nama Kolom": kolom,
            "Tipe Data Sistem": str(df[kolom].dtype),
            "Jumlah Missing Value": df[kolom].isna().sum(),
            "Persentase Missing (%)": round(
                df[kolom].isna().mean() * 100, 2
            ),
            "Jumlah Nilai Unik": df[kolom].nunique(),
            "Sampel Nilai": df[kolom].dropna().astype(str).head(3).tolist()
        })

    return pd.DataFrame(hasil)

In [72]:
df_kamus_data = buat_kamus_data(df_master_analisis)

df_kamus_data

,Nama Kolom,Tipe Data Sistem,Jumlah Missing Value,Persentase Missing (%),Jumlah Nilai Unik,Sampel Nilai
0,order_id,object,0,0.0,10,"[ORD001, ORD002, ORD003]"
1,customer_id,object,0,0.0,10,"[C001, C002, C003]"
2,order_date,datetime64[ns],0,0.0,6,"[2026-09-25, 2026-09-26, 2026-09-26]"
3,total_amount,int64,0,0.0,10,"[150000, 225000, 175000]"
4,coupon_code,object,4,40.0,1,"[AKHIRBULAN, AKHIRBULAN, AKHIRBULAN]"
5,customer_name,object,1,10.0,9,"[Andi, Budi, Citra]"
6,membership_level,object,1,10.0,3,"[Gold, Silver, Platinum]"
7,is_active,float64,1,10.0,1,"[1.0, 1.0, 1.0]"
8,courier,object,3,30.0,4,"[JNE, SiCepat, J&T]"
9,tracking_number,object,3,30.0,7,"[JNE001, SCP002, JNT003]"


In [73]:
df_kamus_data.to_csv(
    "kamus_data_hasil.csv",
    index=False
)

print("kamus_data_hasil.csv berhasil dibuat!")

kamus_data_hasil.csv berhasil dibuat!


In [74]:
# Validasi 1: order_id harus unik
assert df_master_analisis["order_id"].is_unique, \
    "Terdapat duplikasi pada order_id!"

print("✅ Validasi primary key berhasil: tidak ada duplikasi order_id.")

✅ Validasi primary key berhasil: tidak ada duplikasi order_id.


In [75]:
# Validasi 2: total_amount tidak boleh negatif
assert (df_master_analisis["total_amount"] >= 0).all(), \
    "Terdapat total_amount bernilai negatif!"

print("✅ Validasi total_amount berhasil: tidak ada nilai negatif.")

✅ Validasi total_amount berhasil: tidak ada nilai negatif.


In [76]:
# Validasi 3: order_date tidak boleh melebihi tanggal hari ini
assert (
    df_master_analisis["order_date"] <= pd.Timestamp.today()
).all(), \
    "Terdapat tanggal pesanan di masa depan!"

print("✅ Validasi tanggal berhasil: tidak ada tanggal pesanan di masa depan.")

✅ Validasi tanggal berhasil: tidak ada tanggal pesanan di masa depan.


## KESIMPULAN

Pada proses data wrangling ini, data diperoleh dari tiga sumber yang berbeda,
yaitu file CSV transaksi dari Divisi Kasir/POS, database SQLite dari Divisi
CRM/Membership, dan data pengiriman dari REST API simulasi.

Ketiga sumber data berhasil diekstraksi dan diintegrasikan menggunakan Python.
Data transaksi digunakan sebagai data utama, kemudian digabungkan dengan data
pelanggan aktif berdasarkan `customer_id` dan data logistik berdasarkan
`order_id` menggunakan LEFT JOIN.

Selanjutnya, metadata dari data hasil integrasi dibuat secara otomatis dalam
bentuk kamus data dan disimpan sebagai `kamus_data_hasil.csv`. Validasi
kualitas data juga dilakukan menggunakan automated assertions untuk memastikan
tidak terdapat duplikasi `order_id`, nilai `total_amount` negatif, maupun
tanggal pesanan yang melebihi tanggal hari ini.

In [77]:
df_master_analisis

,order_id,customer_id,order_date,total_amount,coupon_code,customer_name,membership_level,is_active,courier,tracking_number,shipping_status
0,ORD001,C001,2026-09-25,150000,AKHIRBULAN,Andi,Gold,1.0,JNE,JNE001,Delivered
1,ORD002,C002,2026-09-26,225000,NaN,Budi,Silver,1.0,SiCepat,SCP002,In Transit
2,ORD003,C003,2026-09-26,175000,AKHIRBULAN,Citra,Platinum,1.0,J&T,JNT003,Delivered
3,ORD004,C004,2026-09-27,320000,AKHIRBULAN,Dina,Gold,1.0,AnterAja,ANT004,Delivered
4,ORD005,C005,2026-09-28,125000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ORD006,C006,2026-09-28,275000,AKHIRBULAN,Fajar,Silver,1.0,JNE,JNE006,In Transit
6,ORD007,C007,2026-09-29,180000,NaN,Gina,Gold,1.0,NaN,NaN,NaN
7,ORD008,C008,2026-09-29,95000,AKHIRBULAN,Hadi,Platinum,1.0,SiCepat,SCP008,Delivered
8,ORD009,C009,2026-09-30,450000,AKHIRBULAN,Intan,Silver,1.0,J&T,JNT009,In Transit
9,ORD010,C010,2026-09-30,210000,NaN,Joko,Gold,1.0,NaN,NaN,NaN


In [78]:
print("Jumlah transaksi:", len(df_master_analisis))

print(
    "Jumlah transaksi menggunakan kupon:",
    df_master_analisis["coupon_code"].notna().sum()
)

print("\nProfil membership pelanggan pengguna kupon:")
print(
    df_master_analisis[
        df_master_analisis["coupon_code"].notna()
    ]["membership_level"].value_counts(dropna=False)
)

print("\nStatus pengiriman:")
print(
    df_master_analisis["shipping_status"].value_counts(dropna=False)
)

Jumlah transaksi: 10
Jumlah transaksi menggunakan kupon: 6

Profil membership pelanggan pengguna kupon:
membership_level
Gold        2
Platinum    2
Silver      2
Name: count, dtype: int64

Status pengiriman:
shipping_status
Delivered     4
In Transit    3
NaN           3
Name: count, dtype: int64
